# 衡东方言 ASR - SenseVoice LoRA 微调与测试

微调 SenseVoice-Small (234M) 模型，并在验证集上评估 CER。

## 1. 环境检查

In [ ]:
import torch
import subprocess
import json
import os

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"显存: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

try:
    import funasr
    print(f"FunASR: {funasr.__version__}")
except ImportError:
    print("FunASR 未安装")

try:
    import torchaudio
    print(f"torchaudio: {torchaudio.__version__}")
except ImportError:
    print("torchaudio 未安装")

## 2. 数据检查

In [ ]:
TRAIN_JSONL = "/mnt/data/prepared_data/sensevoice/train.jsonl"
VAL_JSONL = "/mnt/data/prepared_data/sensevoice/val.jsonl"

for path, label in [(TRAIN_JSONL, '训练集'), (VAL_JSONL, '验证集')]:
    if os.path.exists(path):
        count = sum(1 for _ in open(path))
        print(f"  {label}: {count} 条")
    else:
        print(f"  {label}: 不存在! {path}")

# 音频加载测试
if os.path.exists(TRAIN_JSONL):
    with open(TRAIN_JSONL) as f:
        sample = json.loads(f.readline())
    audio = sample['source']
    print(f"\n音频测试: {audio}")
    if os.path.exists(audio):
        import torchaudio
        wav, sr = torchaudio.load(audio)
        print(f"  OK: shape={wav.shape}, sr={sr}")

## 3. 训练

In [ ]:
OUTPUT_DIR = "/mnt/output/sensevoice_lora"
os.makedirs(OUTPUT_DIR, exist_ok=True)

cmd = [
    "torchrun", "--nproc_per_node=1", "-m", "funasr.bin.train_ds",
    "++model=iic/SenseVoiceSmall",
    f"++train_data_set_list={TRAIN_JSONL}",
    f"++valid_data_set_list={VAL_JSONL}",
    "++dataset_conf.data_split_num=1",
    "++dataset_conf.batch_sampler=BatchSampler",
    "++dataset_conf.batch_size=6000",
    "++dataset_conf.sort_size=1024",
    "++dataset_conf.batch_type=token",
    "++dataset_conf.num_workers=4",
    "++train_conf.max_epoch=10",
    "++train_conf.log_interval=50",
    "++train_conf.validate_interval=2000",
    "++train_conf.save_checkpoint_interval=2000",
    "++train_conf.keep_nbest_models=5",
    "++train_conf.avg_nbest_model=3",
    "++train_conf.use_bf16=true",
    "++train_conf.grad_clip=5",
    "++lora_only=true",
    "++lora_bias=none",
    "++lora_rank=8",
    "++lora_alpha=16",
    "++lora_dropout=0.1",
    "++optim=adamw",
    "++optim_conf.lr=2e-4",
    "++optim_conf.weight_decay=0.0",
    f"++output_dir={OUTPUT_DIR}",
]

print(f"模型: SenseVoiceSmall (LoRA)")
print(f"输出: {OUTPUT_DIR}")
print(f"LR: 2e-4, Epoch: 10")
print()
print("开始训练...")
print("=" * 60)

In [ ]:
import sys

process = subprocess.Popen(
    cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1,
)
for line in process.stdout:
    print(line, end='')
    sys.stdout.flush()

returncode = process.wait()
print(f"\n{'=' * 60}")
if returncode == 0:
    print(f"训练完成! 模型保存在: {OUTPUT_DIR}")
else:
    print(f"训练失败! 返回码: {returncode}")

## 4. 加载模型并测试

In [ ]:
from funasr import AutoModel
import re
import time
import torch

SENSEVOICE_DIR = OUTPUT_DIR

# 查找模型路径
SV_CKPT = f"{SENSEVOICE_DIR}/model.pt.best"
if not os.path.exists(SV_CKPT):
    SV_CKPT = f"{SENSEVOICE_DIR}/models/model.pt.best"
    SENSEVOICE_DIR = f"{SENSEVOICE_DIR}/models"

def clean_sensevoice_text(text):
    """去除特殊标记 <|zh|><|NEUTRAL|>..."""
    return re.sub(r'<\|[^|]*\|>', '', text).strip()

# 加载基座模型 + 微调 checkpoint
print(f"加载基座模型: iic/SenseVoiceSmall (LoRA)")
print(f"加载 checkpoint: {SV_CKPT}")
model = AutoModel(model="iic/SenseVoiceSmall", lora_only=True, disable_update=True)
ckpt = torch.load(SV_CKPT, map_location="cpu")

# 加载权重并检查 key 匹配情况
model_keys = set(model.model.state_dict().keys())
ckpt_keys = set(ckpt["state_dict"].keys())
matched = model_keys & ckpt_keys
missing = model_keys - ckpt_keys
unexpected = ckpt_keys - model_keys
print(f"Key 匹配: {len(matched)}/{len(model_keys)} 模型key, {len(unexpected)} 多余checkpoint key")
if missing:
    print(f"  未匹配 key (前5): {list(missing)[:5]}")
if unexpected:
    print(f"  多余 key (前5): {list(unexpected)[:5]}")

model.model.load_state_dict(ckpt["state_dict"], strict=False)
print("SenseVoice 加载成功!")

# 单条测试
samples = []
with open(VAL_JSONL) as f:
    for i, line in enumerate(f):
        if i >= 10:
            break
        samples.append(json.loads(line))

print(f"\n单条测试 ({len(samples)} 条):")
for s in samples[:5]:
    audio, expected = s['source'], s['target']
    if not os.path.exists(audio):
        continue
    result = model.generate(input=audio, language="auto", use_itn=True)
    pred = clean_sensevoice_text(result[0]['text']) if result else ""
    flag = "V" if expected in pred else "X"
    print(f"  [{flag}] {expected} → {pred}")